In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

In [ ]:
class findsentiment_structure(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='sentiment of the review')

In [ ]:
class diagnosisschema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [ ]:
# model.with_structured_output() is unusable here: the default method='function_calling'
# rejects Pydantic schemas outright, and method='json_schema' sends a TGI-only
# response_format that most providers behind the HF router reject with
# "422 grammar is not valid" roughly half the time.
# So we ask for JSON in the prompt and parse it client-side instead — provider-agnostic,
# and it gives back a real Pydantic object.

def build_structured_chain(schema, task):
    parser = PydanticOutputParser(pydantic_object=schema)
    prompt = PromptTemplate(
        template=task + "\n\n{review}\n\n{format_instructions}",
        input_variables=['review'],
        partial_variables={'format_instructions': parser.get_format_instructions()},
    )
    return prompt | model | parser


structured_model1 = build_structured_chain(
    findsentiment_structure, 'Find the sentiment of the following review.'
)
structured_model2 = build_structured_chain(
    diagnosisschema, 'Diagnose the following negative review.'
)

In [ ]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal['positive', 'negative']
    diagnosis : dict
    response : str
    

In [ ]:
def finesentiment(state: ReviewState):
    sentiment = structured_model1.invoke({'review': state['review']}).sentiment
    
    return {'sentiment': sentiment}

In [ ]:
def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] ==  'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [ ]:
def positive_response(state : ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""

    response = model.invoke(prompt).content
    
    return {'response' : response}

In [ ]:
def run_diagnosis(state : ReviewState):
    diagnosis = structured_model2.invoke({'review': state['review']})
    
    return {'diagnosis': diagnosis.model_dump()}

In [ ]:
def negative_response(state : ReviewState):
    diagnosis = state['diagnosis']
    
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message."""

    response = model.invoke(prompt).content
    
    return {'response' : response}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('finesentiment', finesentiment)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('positive_response', positive_response)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'finesentiment')
graph.add_conditional_edges('finesentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [ ]:
initial_state = {'review': 'The app keeps crashing every time I try to upload a photo. I have lost work three times today and support has not replied.'}

final_state = workflow.invoke(initial_state)

print(final_state['sentiment'])
print(final_state['diagnosis'])
print(final_state['response'])